## Setup — register Gold tables as Spark SQL views

In [0]:
CATALOG = "workspace"
SCHEMA = "transfermarkt"

for t in ["fact_appearance", "dim_date", "dim_competition"]:
    spark.table(f"workspace.gold.{t}").createOrReplaceTempView(t)

In [0]:
%pip install clickhouse-connect --quiet

In [0]:
%restart_python

In [0]:
import clickhouse_connect

client = clickhouse_connect.get_client(
    host=dbutils.secrets.get(scope="clickhouse", key="host"),
    port=8443,
    username=dbutils.secrets.get(scope="clickhouse", key="user"),
    password=dbutils.secrets.get(scope="clickhouse", key="password"),
    secure=True,
)
print(client.command("SELECT 1"))

## Spark SQL — 5 runs

In [0]:
import time

spark_query = """
SELECT dc.name AS competition_name, dd.year,
       SUM(fa.goals) AS total_goals,
       SUM(fa.assists) AS total_assists,
       COUNT(*) AS total_appearances
FROM fact_appearance fa
JOIN dim_date dd ON fa.date_key = dd.date_key
JOIN dim_competition dc ON fa.competition_key = dc.competition_key
GROUP BY dc.name, dd.year
ORDER BY dc.name, dd.year
"""

spark_times = []
for i in range(5):
    start = time.time()
    result = spark.sql(spark_query).collect()
    elapsed = time.time() - start
    spark_times.append(elapsed)
    print(f"Spark run {i+1}: {elapsed:.3f}s ({len(result)} rows)")

print(f"\nSpark cold (run 1): {spark_times[0]:.3f}s")
print(f"Spark steady-state (median of runs 2-5): {sorted(spark_times[1:])[len(spark_times[1:])//2]:.3f}s")

## ClickHouse — same query, same 5-run methodology

In [0]:
clickhouse_query = """
SELECT dc.name AS competition_name, dd.year,
       SUM(fa.goals) AS total_goals,
       SUM(fa.assists) AS total_assists,
       COUNT(*) AS total_appearances
FROM transfermarkt.fact_appearance fa
JOIN transfermarkt.dim_date dd ON fa.date_key = dd.date_key
JOIN transfermarkt.dim_competition dc ON fa.competition_key = dc.competition_key
GROUP BY dc.name, dd.year
ORDER BY dc.name, dd.year
"""

ch_times = []
for i in range(5):
    start = time.time()
    result = client.query(clickhouse_query)
    elapsed = time.time() - start
    ch_times.append(elapsed)
    print(f"ClickHouse run {i+1}: {elapsed:.3f}s ({len(result.result_rows)} rows)")

print(f"\nClickHouse cold (run 1): {ch_times[0]:.3f}s")
print(f"ClickHouse steady-state (median of runs 2-5): {sorted(ch_times[1:])[len(ch_times[1:])//2]:.3f}s")

## Correctness check before trusting any timing
A speed comparison is worthless if the two systems computed different things — checking row count and grand totals agree is not optional here.

In [0]:
spark_result_df = spark.sql(spark_query)
print("Spark rows:", spark_result_df.count())
spark_result_df.selectExpr("sum(total_goals)", "sum(total_assists)", "sum(total_appearances)").show()

ch_result = client.query(clickhouse_query)
print("ClickHouse rows:", len(ch_result.result_rows))
import pandas as pd
ch_pdf = pd.DataFrame(ch_result.result_rows,
                       columns=["competition_name", "year", "total_goals", "total_assists", "total_appearances"])
print(ch_pdf[["total_goals", "total_assists", "total_appearances"]].sum())

## Contrasting benchmark: single-player point lookup
Same 5-run methodology as the aggregation benchmark, same correctness-before-speed discipline.

In [0]:
CATALOG = "workspace"
SCHEMA = "transfermarkt"

spark.table("workspace.gold.fact_player_valuation").createOrReplaceTempView("fact_player_valuation")

player_id_to_check = 17911

spark_point_query = f"""
SELECT date_key, market_value_in_eur
FROM fact_player_valuation
WHERE player_key = {player_id_to_check}
ORDER BY date_key
"""

spark_point_times = []
for i in range(5):
    start = time.time()
    result = spark.sql(spark_point_query).collect()
    elapsed = time.time() - start
    spark_point_times.append(elapsed)
    print(f"Spark run {i+1}: {elapsed:.4f}s ({len(result)} rows)")

print(f"\nSpark steady-state median: {sorted(spark_point_times[1:])[1:3][0] if len(spark_point_times[1:]) % 2 == 0 else sorted(spark_point_times[1:])[len(spark_point_times[1:])//2]:.4f}s")

## Same point lookup, ClickHouse

In [0]:
clickhouse_point_query = f"""
SELECT date_key, market_value_in_eur
FROM transfermarkt.fact_player_valuation
WHERE player_key = {player_id_to_check}
ORDER BY date_key
"""

ch_point_times = []
for i in range(5):
    start = time.time()
    result = client.query(clickhouse_point_query)
    elapsed = time.time() - start
    ch_point_times.append(elapsed)
    print(f"ClickHouse run {i+1}: {elapsed:.4f}s ({len(result.result_rows)} rows)")

sorted_rest = sorted(ch_point_times[1:])
median_ch = (sorted_rest[1] + sorted_rest[2]) / 2
print(f"\nClickHouse steady-state median: {median_ch:.4f}s")

## Correctness check

In [0]:
spark_pdf = spark.sql(spark_point_query).toPandas()
ch_result = client.query(clickhouse_point_query)
ch_pdf = pd.DataFrame(ch_result.result_rows, columns=["date_key", "market_value_in_eur"])

print("Spark rows:", len(spark_pdf), " ClickHouse rows:", len(ch_pdf))
print("Spark values match ClickHouse:", spark_pdf["market_value_in_eur"].tolist() == ch_pdf["market_value_in_eur"].tolist())